In [ ]:
!pip install timm transformers datasets accelerate --quiet

In [ ]:
!pip install shap --quiet

### What is SHAP?

SHAP (SHapley Additive exPlanations) is a game-theoretic approach to explain the output of any machine learning model. It connects optimal credit allocation with local explanations using the classic Shapley values from game theory.

For an image classification model, SHAP can tell us which parts (pixels or superpixels) of an input image contributed most positively or negatively to a specific class prediction. This helps in understanding the model's decision-making process, identifying potential biases, and building trust in the model.

To apply SHAP to our hybrid model, we'll need to:
1.  **Define a prediction function**: This function will take raw images as input, pass them through the `HybridFeatureExtractor`, and then use the trained XGBoost model to get class probabilities.
2.  **Select background data**: SHAP's `KernelExplainer` needs a representative background dataset to understand the typical feature values and compute baselines.
3.  **Select samples for explanation**: We'll pick a few images from the test set to visualize their SHAP explanations.

In [1]:
import zipfile
import os
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
zip_path="/content/drive/MyDrive/Research_project/TrashNeXt Dataset.zip"
extract_path = "/content"


In [3]:
os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted to:", extract_path)
!ls {extract_path}

Dataset extracted to: /content
 drive	 __MACOSX   sample_data  'TrashNeXt Dataset'


In [4]:
import os
from PIL import Image
from tqdm import tqdm

def is_corrupted_image(file_path):
    """Check if an image file is corrupted."""
    try:
        with Image.open(file_path) as img:
            img.verify()  # Verify integrity
        return False
    except (IOError, SyntaxError, Image.DecompressionBombError):
        return True

def remove_corrupted_images(dataset_path):
    """Scan and remove corrupted images + hidden macOS files."""
    corrupted_count = 0
    total_images = 0

    for root, _, files in os.walk(dataset_path):
        for file in tqdm(files, desc=f"Scanning {os.path.basename(root)}"):

            # Skip hidden files (.DS_Store, ._files)
            if file.startswith("."):
                file_path = os.path.join(root, file)
                try:
                    os.remove(file_path)
                    corrupted_count += 1
                    print(f"Removed hidden file: {file_path}")
                except:
                    pass
                continue

            # Check only image formats
            if file.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.gif')):
                total_images += 1
                file_path = os.path.join(root, file)
                if is_corrupted_image(file_path):
                    try:
                        os.remove(file_path)
                        corrupted_count += 1
                        print(f"Removed corrupted image: {file_path}")
                    except Exception as e:
                        print(f"Error removing {file_path}: {e}")

    print("\n✅ Scan completed!")
    print(f"📷 Total images scanned: {total_images}")
    print(f"🗑️ Corrupted/hidden files removed: {corrupted_count}")

# Run the cleaner
dataset_path = '/content/dataset'  # Your dataset path
remove_corrupted_images(dataset_path)


Scanning dataset: 100%|██████████| 1/1 [00:00<00:00, 4882.78it/s]


Removed hidden file: /content/dataset/.DS_Store


Scanning Test: 100%|██████████| 1/1 [00:00<00:00, 4559.03it/s]


Removed hidden file: /content/dataset/Test/.DS_Store


Scanning Train: 100%|██████████| 1/1 [00:00<00:00, 5121.25it/s]


Removed hidden file: /content/dataset/Train/.DS_Store


Scanning Valid: 100%|██████████| 1/1 [00:00<00:00, 5660.33it/s]


Removed hidden file: /content/dataset/Valid/.DS_Store


Scanning metal: 100%|██████████| 258/258 [00:00<00:00, 984.98it/s]


✅ Scan completed!
📷 Total images scanned: 23625
🗑️ Corrupted/hidden files removed: 4


In [5]:
from PIL import Image
import os

def remove_truncated_images(dataset_path):
    for root, _, files in os.walk(dataset_path):
        for file in files:
            if file.lower().endswith(('.jpg', '.jpeg', '.png')):
                try:
                    img_path = os.path.join(root, file)
                    img = Image.open(img_path)
                    img.verify()
                except Exception as e:
                    print(f"Removing corrupted/truncated: {img_path}")
                    os.remove(img_path)

remove_truncated_images("/content/dataset")


In [6]:
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

In [7]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import timm
import numpy as np
import os
from tqdm import tqdm

In [8]:
class HybridFeatureExtractor(nn.Module):
    def __init__(self):
        super(HybridFeatureExtractor, self).__init__()
        # EfficientNetV2 feature extractor
        self.efficientnet = models.efficientnet_v2_s(weights="IMAGENET1K_V1")
        self.efficientnet.classifier = nn.Identity()

        # LeViT feature extractor
        self.levit = timm.create_model('levit_256', pretrained=True)

        # Replace classification heads with identity
        self.levit.head = nn.Identity()
        if hasattr(self.levit, 'dist_head'):
            self.levit.dist_head = nn.Identity()

        # ✅ Force LeViT to skip forward_head completely
        def forward_override(x):
            # run backbone (patch embedding + transformer encoder + pooling)
            x = self.levit.forward_features(x)   # [B, N, D]
            # take CLS token only (LeViT uses CLS for classification)
            x = x[:, 0, :]                       # [B, D]
            return x
        self.levit.forward = forward_override

    def forward(self, x):
        # Resize for each model
        x_eff = transforms.functional.resize(x, (300, 300))
        x_levit = transforms.functional.resize(x, (224, 224))

        with torch.no_grad():
            features_eff = self.efficientnet(x_eff)  # [B, 1280]
            features_levit = self.levit(x_levit)     # [B, 512]

        return torch.cat((features_eff, features_levit), dim=1)


In [9]:
def extract_features(data_loader, model, device):
    all_features = []
    all_labels = []

    model.eval()
    for images, labels in tqdm(data_loader, desc="Extracting features"):
        images = images.to(device)
        features = model(images)
        all_features.append(features.cpu().numpy())
        all_labels.append(labels.cpu().numpy())

    return np.concatenate(all_features), np.concatenate(all_labels)

In [10]:
if __name__ == '__main__':
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # --- Data Loading ---
    # ❗️ IMPORTANT: Update these paths to your dataset location in Google Drive
    train_dir = "/content/dataset/Train" # Corrected path
    valid_dir = "/content/dataset/Valid" # Corrected path

    data_transforms = transforms.Compose([
        transforms.Resize((256, 256)), # Initial resize
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    train_dataset = datasets.ImageFolder(train_dir, transform=data_transforms)
    valid_dataset = datasets.ImageFolder(valid_dir, transform=data_transforms)

    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=False)
    valid_loader = DataLoader(valid_dataset, batch_size=32, shuffle=False)

    # --- Initialize model and extract ---
    feature_extractor = HybridFeatureExtractor().to(device)

    X_train, y_train = extract_features(train_loader, feature_extractor, device)
    X_valid, y_valid = extract_features(valid_loader, feature_extractor, device)

    print(f"Training features shape: {X_train.shape}")
    print(f"Validation features shape: {X_valid.shape}")

    # Save features for the next stage
    np.save('X_train.npy', X_train)
    np.save('y_train.npy', y_train)
    np.save('X_valid.npy', X_valid)
    np.save('y_valid.npy', y_valid)
    print("✅ Features saved!")

Downloading: "https://download.pytorch.org/models/efficientnet_v2_s-dd5fe13b.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_v2_s-dd5fe13b.pth


100%|██████████| 82.7M/82.7M [00:00<00:00, 196MB/s]


model.safetensors:   0%|          | 0.00/75.9M [00:00<?, ?B/s]

Extracting features:   0%|          | 0/591 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
Extracting features: 100%|██████████| 74/74 [00:51<00:00,  1.43it/s]


Training features shape: (18898, 1792)
Validation features shape: (2363, 1792)
✅ Features saved!


In [11]:
import numpy as np
import xgboost as xgb
from sklearn.metrics import accuracy_score, log_loss
import matplotlib.pyplot as plt

# --- Load extracted features ---
X_train = np.load("X_train.npy")
y_train = np.load("y_train.npy")
X_valid = np.load("X_valid.npy")
y_valid = np.load("y_valid.npy")

In [12]:
# Create DMatrix (optimized format for XGBoost)
dtrain = xgb.DMatrix(X_train, label=y_train)
dvalid = xgb.DMatrix(X_valid, label=y_valid)

# --- XGBoost Parameters ---
params = {
    'objective': 'multi:softprob',   # multi-class classification
    'num_class': len(np.unique(y_train)),
    'eval_metric': 'mlogloss',
    'learning_rate': 0.25858285509781914,
    'max_depth': 5,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'seed': 42
}

In [13]:
# --- Training with evaluation ---
num_rounds = 236   # total boosting rounds (like epochs)
evals = [(dtrain, 'train'), (dvalid, 'valid')]
evals_result = {}

model = xgb.train(
    params,
    dtrain,
    num_boost_round=num_rounds,
    evals=evals,
    evals_result=evals_result,
    verbose_eval=1
)

# --- Collect metrics ---
train_logloss = evals_result['train']['mlogloss']
valid_logloss = evals_result['valid']['mlogloss']

# Accuracy at each round
train_acc, valid_acc = [], []
for i in range(1, num_rounds+1):
    y_train_pred = np.argmax(model.predict(dtrain, iteration_range=(0, i)), axis=1)
    y_valid_pred = np.argmax(model.predict(dvalid, iteration_range=(0, i)), axis=1)
    train_acc.append(accuracy_score(y_train, y_train_pred))
    valid_acc.append(accuracy_score(y_valid, y_valid_pred))

    print(f"Round {i}/{num_rounds} "
          f"| Train Loss: {train_logloss[i-1]:.4f}, Train Acc: {train_acc[-1]:.4f} "
          f"| Val Loss: {valid_logloss[i-1]:.4f}, Val Acc: {valid_acc[-1]:.4f}")


[0]	train-mlogloss:1.58673	valid-mlogloss:1.63055
[1]	train-mlogloss:1.29569	valid-mlogloss:1.37450
[2]	train-mlogloss:1.09401	valid-mlogloss:1.19047
[3]	train-mlogloss:0.94645	valid-mlogloss:1.05706
[4]	train-mlogloss:0.82900	valid-mlogloss:0.94868
[5]	train-mlogloss:0.73594	valid-mlogloss:0.86450
[6]	train-mlogloss:0.65777	valid-mlogloss:0.79500
[7]	train-mlogloss:0.59336	valid-mlogloss:0.73720
[8]	train-mlogloss:0.53886	valid-mlogloss:0.69080
[9]	train-mlogloss:0.49119	valid-mlogloss:0.65048
[10]	train-mlogloss:0.44990	valid-mlogloss:0.61627
[11]	train-mlogloss:0.41302	valid-mlogloss:0.58514
[12]	train-mlogloss:0.38084	valid-mlogloss:0.55955
[13]	train-mlogloss:0.35234	valid-mlogloss:0.53834
[14]	train-mlogloss:0.32759	valid-mlogloss:0.52097
[15]	train-mlogloss:0.30484	valid-mlogloss:0.50296
[16]	train-mlogloss:0.28455	valid-mlogloss:0.48867
[17]	train-mlogloss:0.26592	valid-mlogloss:0.47605
[18]	train-mlogloss:0.24890	valid-mlogloss:0.46352
[19]	train-mlogloss:0.23308	valid-mloglos

In [14]:


import time

# --- Define the path to your test data ---
test_dir = "/content/dataset/Test"

# --- Create the test dataset and dataloader ---
test_dataset = datasets.ImageFolder(test_dir, transform=data_transforms)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# --- Extract and save features for the test set ---
print("Extracting features from the test set...")
start_time = time.time()

X_test, y_test = extract_features(test_loader, feature_extractor, device)

end_time = time.time()
print(f"Feature extraction for test set took: {end_time - start_time:.2f} seconds")

print(f"Test features shape: {X_test.shape}")

# --- Save features for later use ---
np.save('X_test.npy', X_test)
np.save('y_test.npy', y_test)
print("✅ Test features saved!")

Extracting features from the test set...


Extracting features:   1%|▏         | 1/74 [00:00<00:45,  1.59it/s]/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
Extracting features: 100%|██████████| 74/74 [00:49<00:00,  1.50it/s]

Feature extraction for test set took: 49.27 seconds
Test features shape: (2364, 1792)
✅ Test features saved!


In [15]:


import numpy as np
import time
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, log_loss

# --- Load the test set features ---
X_test = np.load("X_test.npy")
y_test = np.load("y_test.npy")

# --- Create the DMatrix for the test set ---
dtest = xgb.DMatrix(X_test, label=y_test)

# --- Helper function to calculate TPR and FPR from a confusion matrix ---
def calculate_tpr_fpr(cm):
    """Calculates macro-average TPR and FPR for a multi-class confusion matrix."""
    num_classes = cm.shape[0]
    tpr_list = []
    fpr_list = []

    for i in range(num_classes):
        tp = cm[i, i]
        fn = np.sum(cm[i, :]) - tp
        fp = np.sum(cm[:, i]) - tp
        tn = np.sum(cm) - (tp + fn + fp)

        # True Positive Rate (Recall)
        tpr = tp / (tp + fn) if (tp + fn) > 0 else 0.
        tpr_list.append(tpr)

        # False Positive Rate
        fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.
        fpr_list.append(fpr)

    return np.mean(tpr_list), np.mean(fpr_list)


# ===================================================================
#                       PERFORMANCE EVALUATION
# ===================================================================

# Get the class names from the dataset folder structure
class_names = train_dataset.classes

# --- Get predictions (probabilities and final labels) ---
# Training Set
train_start_time = time.time()
y_train_pred_proba = model.predict(dtrain)
train_pred_time = time.time() - train_start_time
y_train_pred = np.argmax(y_train_pred_proba, axis=1)

# Validation Set
valid_start_time = time.time()
y_valid_pred_proba = model.predict(dvalid)
valid_pred_time = time.time() - valid_start_time
y_valid_pred = np.argmax(y_valid_pred_proba, axis=1)

# Testing Set
test_start_time = time.time()
y_test_pred_proba = model.predict(dtest)
test_pred_time = time.time() - test_start_time
y_test_pred = np.argmax(y_test_pred_proba, axis=1)


# --- Calculate Metrics ---
# Training
report_train = classification_report(y_train, y_train_pred, target_names=class_names, output_dict=True)
cm_train = confusion_matrix(y_train, y_train_pred)
tpr_train, fpr_train = calculate_tpr_fpr(cm_train)
loss_train = evals_result['train']['mlogloss'][-1] # Get final loss from training history

# Validation
report_valid = classification_report(y_valid, y_valid_pred, target_names=class_names, output_dict=True)
cm_valid = confusion_matrix(y_valid, y_valid_pred)
tpr_valid, fpr_valid = calculate_tpr_fpr(cm_valid)
loss_valid = evals_result['valid']['mlogloss'][-1] # Get final loss from training history

# Testing
report_test = classification_report(y_test, y_test_pred, target_names=class_names, output_dict=True)
cm_test = confusion_matrix(y_test, y_test_pred)
tpr_test, fpr_test = calculate_tpr_fpr(cm_test)
loss_test = log_loss(y_test, y_test_pred_proba) # Calculate test loss manually
auroc_test = roc_auc_score(y_test, y_test_pred_proba, multi_class='ovr', average='weighted')


# --- Print Final Reports ---

print("="*50)
print("          TRAINING SET METRICS")
print("="*50)
print(f"Accuracy:          {report_train['accuracy']:.4f}")
print(f"Weighted Precision:  {report_train['weighted avg']['precision']:.4f}")
print(f"Weighted Recall/TPR: {report_train['weighted avg']['recall']:.4f}")
print(f"Weighted F1-Score:   {report_train['weighted avg']['f1-score']:.4f}")
print(f"Macro Avg FPR:       {fpr_train:.4f}")
print(f"Final Loss:          {loss_train:.4f}")
# Note: Training time is the time for the entire xgb.train() call
print(f"Inference Time:      {train_pred_time:.4f} seconds\n")


print("="*50)
print("         VALIDATION SET METRICS")
print("="*50)
print(f"Accuracy:          {report_valid['accuracy']:.4f}")
print(f"Weighted Precision:  {report_valid['weighted avg']['precision']:.4f}")
print(f"Weighted Recall/TPR: {report_valid['weighted avg']['recall']:.4f}")
print(f"Weighted F1-Score:   {report_valid['weighted avg']['f1-score']:.4f}")
print(f"Macro Avg FPR:       {fpr_valid:.4f}")
print(f"Final Loss:          {loss_valid:.4f}")
print(f"Validation Time:     {valid_pred_time:.4f} seconds\n")


print("="*50)
print("           TESTING SET METRICS")
print("="*50)
print(f"Accuracy:          {report_test['accuracy']:.4f}")
print(f"Weighted Precision:  {report_test['weighted avg']['precision']:.4f}")
print(f"Weighted Recall/TPR: {report_test['weighted avg']['recall']:.4f}")
print(f"Weighted F1-Score:   {report_test['weighted avg']['f1-score']:.4f}")
print(f"Macro Avg FPR:       {fpr_test:.4f}")
print(f"Weighted AUROC:      {auroc_test:.4f}")
print(f"Final Loss:          {loss_test:.4f}")
print(f"Testing Time:        {test_pred_time:.4f} seconds\n")

          TRAINING SET METRICS
Accuracy:          0.9999
Weighted Precision:  0.9999
Weighted Recall/TPR: 0.9999
Weighted F1-Score:   0.9999
Macro Avg FPR:       0.0000
Final Loss:          0.0013
Inference Time:      0.0023 seconds

         VALIDATION SET METRICS
Accuracy:          0.9099
Weighted Precision:  0.9099
Weighted Recall/TPR: 0.9099
Weighted F1-Score:   0.9098
Macro Avg FPR:       0.0113
Final Loss:          0.3092
Validation Time:     0.0006 seconds

           TESTING SET METRICS
Accuracy:          0.9052
Weighted Precision:  0.9055
Weighted Recall/TPR: 0.9052
Weighted F1-Score:   0.9053
Macro Avg FPR:       0.0118
Weighted AUROC:      0.9925
Final Loss:          0.3426
Testing Time:        0.0847 seconds



In [16]:
import pandas as pd

# 1. Initialize data list
data = []

# 2. Iterate through each class to get metrics
for cls in class_names:
    # Precision, Recall, F1, Support from classification_report
    metrics = report_test[cls]

    # Calculate Per-class Accuracy
    # Formula: (TP + TN) / Total
    cls_idx = class_names.index(cls)
    tp = cm_test[cls_idx, cls_idx]
    fp = cm_test[:, cls_idx].sum() - tp
    fn = cm_test[cls_idx, :].sum() - tp
    tn = cm_test.sum() - (tp + fp + fn)
    cls_accuracy = (tp + tn) / cm_test.sum()

    data.append({
        'Class': cls,
        'Precision': round(metrics['precision'], 2),
        'Recall': round(metrics['recall'], 2),
        'F1-Score': round(metrics['f1-score'], 2),
        'Support': int(metrics['support']),
        'Accuracy': round(cls_accuracy, 4)
    })

# 3. Create DataFrame
results_df = pd.DataFrame(data)

# 4. Add Weighted Average row
weighted_avg = report_test['weighted avg']
results_df.loc[len(results_df)] = [
    'Weighted avg',
    round(weighted_avg['precision'], 2),
    round(weighted_avg['recall'], 2),
    round(weighted_avg['f1-score'], 2),
    f"{int(weighted_avg['support']):,}",
    round(report_test['accuracy'], 4)
]

# Display the table
display(results_df)

print(f"Note: Per-class values are rounded to 2 decimal places. Weighted average reflects test accuracy of {report_test['accuracy']:.4f}")

,Class,Precision,Recall,F1-Score,Support,Accuracy
0,cardboard,0.88,0.89,0.88,235,0.9767
1,e-waste,0.94,0.96,0.95,301,0.9873
2,foam_rubber,0.87,0.87,0.87,286,0.9683
3,glass,0.93,0.88,0.91,252,0.9805
4,medical,0.90,0.93,0.91,196,0.9852
5,metal,0.90,0.89,0.89,258,0.9767
6,organic,0.97,0.95,0.96,299,0.9898
7,paper,0.88,0.90,0.89,270,0.9746
8,plastic,0.88,0.87,0.87,267,0.9712
9,Weighted avg,0.91,0.91,0.91,"2,364",0.9052


Note: Per-class values are rounded to 2 decimal places. Weighted average reflects test accuracy of 0.9052
